# Symbolic calculation

This notebook contains the exercises of the symbolic calculation lecture part 5.

In [ ]:
import sympy as sym
import math
from typing import List, Dict
import graphviz
import numbers
from abc import ABC,abstractmethod

x,y,z,t,n,c,w = sym.symbols("x y z t n c w")

# AST complex processing

## Exercise 5.1

In this exercise, we will discover how SymPy expressions are represented as AST and how they could be rendered.

* Declare the following expressions with and without by preventing their evaluation/simplification at creation when needed and possible:
  * $x + 2y$
  * $\sin( w * x + b)$
  * $\frac{ x+1} {y+3}$
  * $\sum \limits _{n=1} ^{\infty}\frac{1} {n^4}$
  * $\sum \limits _{n=1} ^{\infty}\frac{1} {\sin^2(n)}$
  * $\frac{d}{dt}(t^2+1)$
  * $\frac{\partial^2}{\partial y \partial x} (xy + e^{xy}) $
  * $\int _{0} ^{1} (x+1)^2 dx$
* For each of them, observe their trees with the srepr, dotprint/graphviz.Source/view, latex functions

## Exercise 5.2

We will experiment with standard polish notation for an AST. Consider that a mathematical expression is a tree (AST) and could be equivalently represented as a sequence with the operator first, followed by the operands:
* X+4 as [+,x, 4]
* \sin(xy) as [\sin,*,x,y]

* Declare a map that gives the corresponding string of a SymPy AST constructor
  * Mul() -> "*"
  * Add() -> "+"
  * ...
* Code a recursive function ast2polish that transforms a SymPy ast as a list of symbols satisfying the polish notation. In case of leaf values (numbers.Number, sym.Integer ...) and symbol, the function returns a singleton list with the input ast unchanged.
  * 1+2 -> ["+", 1,2]
 
Hints:
  * The extend method merges two lists
  * The function isinstance check that an instance is a subtype of a type
  * The instance attribute __class__ returns the type of an instance


* Test it on the following expressions:
  * X + y
  * 4 + yx
  * $\sin( xy + \frac{2} {5} )y$
  * $x^2 + xyz + 5$. Oberve that + and * are Nary operators.

* Transforms x + 2zt and x + 2z + t by preventing the reordering or simplification of the expression (Hint: Add or Mul with parameter evaluate=False)
* Is is satisfying?
* How to solve this problem in case we would like to code the reverse function polish2ast?
* Code a function ast2polish2 that solves the problem

* Optional: think about a way to code the reverse function polish2ast2 of the previous function ast2polish2

## Exercise 5.3

## AST Traversal and processing

* Look at the following definition of the __AstProcessor__ class and its __AstDummyProcessor__ class implementation.

In [ ]:
class AstProcessor:
    
    def process_node (self, node, parameter = None):
        methodRef = self.process_default
        if isinstance(node,numbers.Number):
            methodRef = self.process_Number
        else:
            try:
                methodName = "process_" + node.func.__name__
                methodRef = getattr(self, methodName)
            except AttributeError:
                methodRef = self.process_default
        return methodRef(node, parameter)

    def process_args (self, node, parameter=None) -> List[object]:
        return [ pa for a in node.args 
                for pa in [self.process_node(a, parameter)]]

    def process_default (self, node, parameter):
        print("process_default called", sym.srepr(node))
        return node

    def process_Number (self, node, parameter):
        print("process_Number called", node)
        return node


class AstDummyProcessor (AstProcessor):

    def process_Add(self, node, parameter:bool ):
        return sym.Mul( *node.args)

    def process_default (self, node, parameter:bool ):
        return node


We will code a subclass of AstProcessor named Evaluator that, given a valuation, i.e, a map that returns a value for every variable symbol occurring in the expression, evaluate the AST and returns its value. For example,
  * Evaluator.process_node( x+y , {x:1,y:2} ) returns 5.
In this exercise, we will not use the SymPy substitution subs function but try to code it in order to understand how it works.

* Code the needed methods (process_add,...) for evaluating $x-2*y$
* Test the evaluator with the {x:1, y:2} valuation.

* Code the required methods in ExtendedEvaluator for evaluating the following expression:
  * $ \frac{x} {y} * \sin ( x^3 ) - 4$

## Exercise 5.4

## Derivation and processing

In this exercise, we will code a processor that calculates the derivative of an expression without using the derivative set of derivation functions in SymPy. The differentiation variable will be given when the derivator processor is instantiated.

* Code a subclass of __ASTProcessor__ (see Exercise 5.2) named __Derivator__ with a constructor setting the derivative variable
* Code a derive and derive_args methods that take an expression and returns the derivative expression according to the derivative variable
* Code the required methods in the Derivator class for calculating the basic derivation according to x of the following expressions:
  * $10$ (process_Number methods)
  * $x$ (process_Symbol method)
  * $y$
  * $x + 6$ (process_Add function)


The SymPy AST node is composed of 2 attributes, __func__ (constructor of node) and __args__ (list of sub-nodes). Recall the following properties:
* Node.func( *node.args ) = node
* $\frac{df(x)g(x)} {dx}  = \frac{d f(x)} {dx} g(x) + f(x) \frac{d g(x)} {dx}$
* The duplication of a Python list or tuple is performed by list(myListOrTuple)

* Code the required method of the __Derivator__ class for derivating $3x$ expression.

Hints:
* Collect the derived arguments calculated using process_args method
* Iterate on every argument of the node:
  * Copy arguments in a mulArgs list
  * Change the ith element of mulArgs with the derivative of the corresponding argument of the node
  * Create a multiplicative term and append it in a addArgs list
* Use the addArgs to create a returned  addition term

Code the required methods of the Derivator class for derivating the following expressions.
  * $2x + x^2$
  * $x \sin ( 3 x )$
  * $\frac{x + y} {x^2 - 1}$
  * $(1+\sin(x) )^2$

## Exercise 5.5

In this exercise, we will introduce heuristics for antiderivative calculation as a processor.

* Look at the following classes definition:
  * INtegrator is an abstract class
  * CompositeIntegrator is based on a set of Integrator. Look at its find_antiderivative method
  * PrimitiveSearcherBase is the base class for every heuristic antiderivative searcher that will be implemented hereafter.


In [ ]:
class Integrator(ABC):

    @abstractmethod
    def findPrimitive( self, ast: object) -> object:
        pass
    

class CompositeIntegrator(Integrator):

    def __init__(self, variable: sym.Symbol):
        self.subIntegrators = set()
        self.variable = variable

    def declareSubIntegrator(self, integrator:Integrator):
        self.subIntegrators.add(integrator)
    
    def findPrimitive( self, ast: object) -> object:
        for s in self.subIntegrators:
            primitive = s.findPrimitive(ast)
            if primitive is not None:
                return primitive
        return None

class IntegratorBase(Integrator, AstProcessor):
    
    def __init__(self, compositeIntegrator: CompositeIntegrator):
        self.compositeIntegrator = compositeIntegrator
        self.compositeIntegrator.declareSubIntegrator(self)
        self.variable = compositeIntegrator.variable

    @abstractmethod
    def isConstant( self, ast: object) -> bool:
        pass
    
    def findPrimitive(self, ast: object) -> object:
        return self.process_node(ast)

    def process_default (self, node, parameter):
        return None
        


* Implement a __FundamentalIntegrator__ class inheriting from IntegratorBase that returns the antiderivative of fundamental functions: p, \sin(x), \cos(x), x, x**p  (p constant)
* The constuctor of __FundamentalIntegrator__ has a compositeIntegrator input parameter of type __CompositeIntegrator__. It calls the upper __IntegratorBase__ constructor in its body(hint: IntegratorBase.\__init__(self,compositeIntegrator))
* What is a constant expression when a derivation is calculated against a variable?
* Overload the isConstant method (look at its declaration in the __IntegratorBase__) that determines if an ast is constant. 
* Build a compositeIntegrator instance based on the x variable (dummy integration variables)
* Build a fundamentalIntegrator instance
* Try the compositeIntegrator and its find_antiderivative method with:
  * X
  * \sin(x)
  * X**5



The __LinearIntegrator__ will take care of the linear properties of the integration operator:
  * $ \int \lambda f(x)dx = \lambda \int f(x)dx $
  * $\int (f(x)+g(x))dx = \int f(x)dx + \int g(x)dx$

* Implement the __LinearIntegrator__ class by following the same pattern than the __FundamentalIntegrator__ class for the inheritance and the constructor
* Implement an instance method isConstant that determines if an ast is constant (look herebefore)
* If f(x) follows the pattern a * f(x) where a is a constant (method process_Mul), deduce the primitive by applying the previous linear properties (Hint: use isConstant method)
* If $f(x) = g_1(x) + ...+ g_n(x)$ (method process_Add), deduce the primitive by applying the previous linear properties





* Build a compositeIntegrator instance based on the x variable (dummy integration variables)
* Build a FundamentaIntegrator instance
* Build a LinearIntegrator instance
* Try the compositeIntegrator and its find_antiderivative method with:
  * 3*x
  * \sin(x) + x
  * 3*\sin(x) + 2*x


* Think about coding an heuristic class about changing of dummy variable
* Think about coding code an heuristic class about integration by part
* Propose different expressions for testing those heuristics


## Changing of variable

## Integration by parts

* What could you envision other integration heuristics?